# RL01 · PPO Portfolio Rebalancing

**Paper:** Liu, X., et al. (2021). FinRL: A deep reinforcement learning library for automated stock trading. *ICAIF 2021*.

This notebook demonstrates training a trading agent using the `StockTradingEnv` Gym-compatible environment.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from empirlab.rl.envs.stock_env import StockTradingEnv
np.random.seed(42)

## 1 · Generate Synthetic Price Data

In [ ]:
n_steps = 252  # one trading year
rng = np.random.default_rng(42)
# Geometric Brownian Motion
dt = 1/252; mu = 0.10; sigma = 0.20; S0 = 100.0
log_returns = (mu - 0.5*sigma**2)*dt + sigma*np.sqrt(dt)*rng.standard_normal(n_steps)
prices = S0 * np.exp(np.cumsum(log_returns))
print(f"Price series: {prices.shape}, range [{prices.min():.1f}, {prices.max():.1f}]")

## 2 · Initialise Environment

In [ ]:
env = StockTradingEnv(
    prices=prices,
    initial_cash=10_000.0,
    transaction_cost=0.001,
    window=10,
)
obs, info = env.reset()
print(f"Observation space: {env.observation_space}")
print(f"Action space:      {env.action_space}")
print(f"Initial obs shape: {obs.shape}")

## 3 · Random Policy Rollout (Baseline)

In [ ]:
obs, info = env.reset()
done = False; portfolio_values = [env.portfolio_value()]
while not done:
    action = env.action_space.sample()   # random
    obs, reward, terminated, truncated, info = env.step(action)
    done = terminated or truncated
    portfolio_values.append(env.portfolio_value())
random_return = portfolio_values[-1] / portfolio_values[0] - 1
print(f"Random policy total return: {random_return:.2%}")

## 4 · Buy-and-Hold Baseline

In [ ]:
obs, info = env.reset()
done = False; bnh_values = [env.portfolio_value()]
step = 0
while not done:
    action = 1 if step == 0 else 0   # buy at start, hold
    obs, reward, terminated, truncated, info = env.step(action)
    done = terminated or truncated
    bnh_values.append(env.portfolio_value()); step += 1
bnh_return = bnh_values[-1] / bnh_values[0] - 1
print(f"Buy-and-hold total return: {bnh_return:.2%}")

## 5 · Plot Equity Curves

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(portfolio_values, label='Random Policy', alpha=0.7)
ax.plot(bnh_values, label='Buy and Hold', lw=2)
ax.set_title('Equity Curves: Random vs Buy-and-Hold')
ax.set_xlabel('Step'); ax.set_ylabel('Portfolio Value ($)')
ax.legend(); plt.tight_layout(); plt.show()

## 6 · Next: Train PPO Agent

The `PPOAgent` in `empirlab.rl.agents` (v0.2) will provide a `fit(env, n_episodes)` API compatible with this environment. For now, use [Stable-Baselines3](https://stable-baselines3.readthedocs.io/) with this Gym environment:

```python
from stable_baselines3 import PPO
model = PPO('MlpPolicy', env, verbose=1)
model.learn(total_timesteps=50_000)
```